In [1]:
import time
from collections import deque
from datetime import datetime

from service_example import getKisToken
from service.hc_stock_minute_pipeline import inquire_time_itemchartprice

from database import getStockMstList, insert_stock_minute2_bulk

In [2]:
KIS_TOKEN = getKisToken()

In [3]:
def getTargetTime():
    """
    크론탭 실행 시간을 기준으로 30분봉 수집이 완료된 시간 return
    10:34 호출 시 103000 출력
    """
    now = datetime.now()

    minute = 30 if now.minute >= 30 else 0

    target = now.replace(
        minute=minute,
        second=0,
        microsecond=0
    )

    return target.strftime("%H%M%S")

In [4]:
class RateLimiter:
    """
    30분 내에 api 호출 완료를 위한 슬라이딩 윈도우 레이트리미터
    무조건 1.2초를 쉬는 기존 코드와 달리 필요시에만 기다리기에 속도상 유리함
    ex)
    0.00초: 호출 → timestamps = [0.00]
    0.06초: 호출 → timestamps = [0.00, 0.06]
    ...
    0.94초: 18번째 호출 → timestamps = [0.00, 0.06, ..., 0.94]  ← 18개 꽉 참
    0.95초: 19번째 호출 시도
            → 가장 오래된 0.00초 기준으로 1.0 - 0.95 = 0.05초 대기
            → 1.00초에 호출, timestamps에서 0.00 제거 후 1.00 추가
    """
    def __init__(self, calls_per_sec=18):
        self.calls_per_sec = calls_per_sec
        self.timestamps = deque() # 최근 호출 시각 저장

    def wait(self):
        now = time.time()
        # 1초보다 오래된 기록 제거
        while self.timestamps and now - self.timestamps[0] >= 1.0:
            self.timestamps.popleft()

        # 1초 안에 18건 이상 쌓였을 경우 대기
        if len(self.timestamps) >= self.calls_per_sec:
            sleep_time = 1.0 - (now - self.timestamps[0])
            if sleep_time > 0:
                time.sleep(sleep_time)

        self.timestamps.append(time.time())

In [5]:
TIMEOUT_MINUTES = 25  # 30분 크론 기준, 25분 안에 강제 종료

def get_itemchartprice_data():
    """
    크론으로 작동되는 코드
    """
    ## api 호출 시간 TIMEOUT_MINUTES 이상으로 넘어가지 못하게끔
    start_time = time.time()
    limiter = RateLimiter(calls_per_sec=18) ## api 호출- 1초에 18번 초과되지 못하게 처리

    try:

        tickersList = getStockMstList() ## 오늘 상장한 종목 리스트 가져오기
        target_time = getTargetTime() ## 호출할 분봉 데이터 시간 계산
        total_len = len(tickersList) ## 상장 종목 갯수

        targetTimes = ["153000", "150000", "143000", "140000", "133000", "130000", "123000", "120000", "113000", "110000", "103000", "100000", "093000"]

        print(f"📌 수집 시작 | 기준시각: {target_time} | 종목 수: {total_len}")

        ## 시간 단위로 호출되는 api이므로 종목을 기준으로 반복문 진행
        for idx, ticker in enumerate(tickersList, 1):

            for target_time in targetTimes:

                ## TIMEOUT_MINUTES 로직
                elapsed = (time.time() - start_time) / 60
                if elapsed >= TIMEOUT_MINUTES:
                    print(f"⏰ {TIMEOUT_MINUTES}분 초과로 강제 종료 (진행: {idx-1}/{total_len})")
                    break

                arr = []
                success = False
                retry_count = 0

                ## 최대 3번까지 시도하고 실패 시 다음 종목으로 이동
                while not success and retry_count < 3:
                    try:
                        limiter.wait()  # 고정 sleep 대신 레이트리미터
                        res = inquire_time_itemchartprice("output2", ticker, target_time, KIS_TOKEN) ## 분봉 호출 api


                        success = True
                    except Exception as api_e:
                        error_msg = str(api_e)

                        if "EGW00201" in error_msg or "초당 거래건수" in error_msg:
                            retry_count += 1
                            print(f"   ⚠️ [{ticker}] 초당 제한. 3초 대기 후 재시도... ({retry_count}/3)")
                            time.sleep(3)

                        else:
                            print(f"   ❌ [{ticker}] 에러: {api_e}")
                            break

                if success and res:
                    for row in res:
                        b_date = f"{row['stck_bsop_date'][:4]}-{row['stck_bsop_date'][4:6]}-{row['stck_bsop_date'][6:]}"
                        c_hour = f"{row['stck_cntg_hour'][:2]}:{row['stck_cntg_hour'][2:4]}:{row['stck_cntg_hour'][4:]}"
                        arr.append((
                            ticker, b_date, c_hour, row['stck_oprc'],
                            row['stck_hgpr'], row['stck_lwpr'], row['stck_prpr'],
                            row['cntg_vol'], row['acml_tr_pbmn']
                        ))

                if arr:
                    try:
                        insert_stock_minute2_bulk(arr)
                        print(f"target_time: [{idx-1}/{total_len}] 완료")
                    except Exception as db_e:
                        print(f"   ❌ [{ticker}] DB 적재 실패: {db_e}")

        elapsed_total = (time.time() - start_time)
        print(f"✅ 수집 완료 | 소요시간: {elapsed_total:.1f}초")

    except Exception as e:
        print(f"🚨 치명적 에러: {e}")

In [6]:
get_itemchartprice_data()

📌 수집 시작 | 기준시각: 160000 | 종목 수: 2733
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[0/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[0/2733] 완료
[1/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[1/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[1/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[2/2733] 완료
[3/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
[3/2733] 완료
[3/2733] 완료
  [retry 1/3] /code 500/ Rate limit, 0.2s 대기
  [retry 2/3] /code 500/ Rate limit, 1.2s 대기
[3/2733] 완료
[3/2733] 완료
[3/2733] 완료
  [retry 1/3] /code